# Student Performance Prediction & Academic Analytics System
### Bachelor of Computer Applications (BCA) - Final Year / Major Project
**Technologies:** Python, Pandas, Seaborn, Matplotlib, Scikit-Learn (Decision Trees), Power BI

---
## 1. Project Overview & Objectives
Student academic success is influenced by complex interactions between past academic foundations, internal assessments, attendance patterns, study habits, and lifestyle factors. This project aims to:
1. Perform comprehensive **Exploratory Data Analysis (EDA)** using **Seaborn** to uncover key determinants of student performance.
2. Train a **Decision Tree Classifier** to classify students into performance tiers (*Distinction, Merit, Pass, At-Risk*).
3. Train a **Decision Tree Regressor** to predict exact final examination percentage scores (0-100).
4. Export visual tree structures and prepare an enriched dataset for **Power BI executive dashboards**.

## 2. Importing Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_absolute_error, root_mean_squared_error, r2_score
)

# Setting plot aesthetics
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
print("All libraries imported successfully!")

## 3. Data Loading and Initial Exploration
We load the cleaned student performance dataset (`data/student_performance_cleaned.csv`).

In [ ]:
df = pd.read_csv("data/student_performance_cleaned.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Checking statistical summary
df.describe()

In [ ]:
# Checking for missing values and data types
df.info()

## 4. Exploratory Data Analysis (EDA) using Seaborn
### 4.1 Correlation Heatmap
Evaluating linear relationships among attendance, test marks, study hours, and final exam outcomes.

In [ ]:
numeric_cols = [
    "Age", "Study_Hours_Per_Week", "Attendance_Rate",
    "Past_Exam_Score", "Internal_Assessment_Score",
    "Assignment_Completion_Rate", "Sleep_Hours_Per_Day", "Final_Score"
]
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Academic & Behavioral Features", fontsize=14, fontweight="bold")
plt.show()

### 4.2 Attendance vs. Final Performance
Examining the correlation between attendance rate and final examination percentage.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Attendance_Rate",
    y="Final_Score",
    hue="Performance_Category",
    palette={"Distinction": "#1f77b4", "Merit": "#2ca02c", "Pass": "#ff7f0e", "At-Risk": "#d62728"},
    alpha=0.8
)
plt.axvline(75, color="gray", linestyle="--", label="75% Mandatory Attendance")
plt.axhline(50, color="red", linestyle="--", label="50% Pass Threshold")
plt.title("Attendance Rate vs. Final Examination Score", fontsize=13, fontweight="bold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.3 Study Hours & Pass/Fail Distribution
Kernel Density Estimation (KDE) comparing study habits of passing vs failing students.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="Study_Hours_Per_Week", hue="Passed", kde=True, bins=25, palette="Set1", alpha=0.5)
plt.title("Distribution of Weekly Study Hours by Passing Status", fontsize=13, fontweight="bold")
plt.xlabel("Study Hours Per Week")
plt.show()

## 5. Feature Engineering & Preprocessing
Encoding categorical variables (`Gender`, `Parental_Education`, `Tutoring_Classes`, etc.) for scikit-learn models.

In [ ]:
feature_cols = [
    "Age", "Gender", "Parental_Education", "Study_Hours_Per_Week",
    "Attendance_Rate", "Past_Exam_Score", "Internal_Assessment_Score",
    "Assignment_Completion_Rate", "Tutoring_Classes", "Internet_Access",
    "Extracurricular_Activities", "Sleep_Hours_Per_Day"
]

X = df[feature_cols].copy()
y_class = df["Performance_Category"].copy()
y_reg = df["Final_Score"].copy()

# Encode categorical columns
encoders = {}
for col in ["Gender", "Parental_Education", "Tutoring_Classes", "Internet_Access", "Extracurricular_Activities"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

# Split for Classification
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Split for Regression
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_c.shape[0]}, Test set size: {X_test_c.shape[0]}")

## 6. Machine Learning: Decision Tree Classifier
We use entropy-based information gain with controlled tree depth to avoid overfitting.

In [ ]:
clf = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)
acc = accuracy_score(y_test_c, y_pred_c)
print(f"Decision Tree Classification Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test_c, y_pred_c))

In [ ]:
# Confusion Matrix Visualization
labels = ["Distinction", "Merit", "Pass", "At-Risk"]
cm = confusion_matrix(y_test_c, y_pred_c, labels=labels)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Decision Tree Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Tier")
plt.ylabel("Actual Tier")
plt.show()

## 7. Machine Learning: Decision Tree Regressor
Predicting continuous numerical final scores (0-100).

In [ ]:
reg = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=6,
    min_samples_split=12,
    min_samples_leaf=6,
    random_state=42
)
reg.fit(X_train_r, y_train_r)

y_pred_r = reg.predict(X_test_r)
print(f"Mean Absolute Error (MAE):  {mean_absolute_error(y_test_r, y_pred_r):.2f} marks")
print(f"Root Mean Squared Error:    {root_mean_squared_error(y_test_r, y_pred_r):.2f} marks")
print(f"R-Squared (R²) Score:       {r2_score(y_test_r, y_pred_r):.4f}")

## 8. Decision Tree Interpretability & Feature Importance

In [ ]:
# Feature Importance Bar Chart
feat_df = pd.DataFrame({
    "Feature": [c.replace("_", " ") for c in feature_cols],
    "Importance": clf.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df, x="Importance", y="Feature", palette="mako", hue="Feature", legend=False)
plt.title("Decision Tree Feature Importances", fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# Plot Decision Tree Structure
plt.figure(figsize=(22, 10), dpi=300)
plot_tree(
    clf,
    feature_names=[c.replace("_", " ") for c in feature_cols],
    class_names=clf.classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title("Decision Tree Flow Diagram (Top Levels)", fontsize=15, fontweight="bold")
plt.show()

## 9. Conclusion & Power BI Integration
- The Decision Tree demonstrated high interpretability while achieving high classification and regression accuracy.
- Past academic history, attendance rate, internal marks, and study hours emerged as the strongest predictors.
- The processed dataset is exported to `data/student_performance_powerbi.csv` for multi-dimensional interactive dashboards in Power BI Desktop.